# Tugas 1 - Prediksi Customer Churn dengan Artificial Neural Network (TensorFlow / Keras)

| Nama | NRP |
|------|-----|
| Catherine Aprilia Harlina | 5054251012 |

## Deskripsi Tugas
Pada tugas ini, kita akan menggunakan Churn Modelling Dataset. Dataset bisa diakses melalui link berikut:\
https://www.kaggle.com/datasets/shrutimechlearn/churn-modelling

Dataset berisi data 10.000 nasabah bank, seperti skor kredit, negara, usia, saldo, dan status keaktifan. Tujuan utama dari tugas ini adalah membangun model Artificial Neural Network (ANN) menggunakan **TensorFlow / Keras** untuk memprediksi apakah seorang nasabah akan berhenti menjadi nasabah (churn, kolom `Exited`) atau tidak.

Langkah-langkah yang harus dilakukan antara lain:
1. Persiapan Dataset dan Eksplorasi Awal
   - Memuat dataset, melihat struktur data, dan distribusi label.
2. Preprocessing
   - Memproses data agar siap digunakan oleh ANN.
3. Eksperimen Model
   - Bangun model baseline Logistic Regression dan beberapa konfigurasi ANN.
4. Evaluasi Model
   - Hitung metrik evaluasi dan visualisasikan hasilnya.
5. Analisis dan Kesimpulan
   - Bandingkan hasil seluruh model dan berikan kesimpulan.

# 1. Persiapan Dataset dan Eksplorasi Awal

In [ ]:
!pip install tensorflow


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


: 

In [ ]:
# Import library yang dibutuhkan (pandas, numpy, matplotlib, seaborn, sklearn, tensorflow, dll)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

# Set random seed agar hasil eksperimen dapat direproduksi, gunakan tf.keras.utils.set_random_seed(42)
tf.keras.utils.set_random_seed(42)
np.random.seed(42)

In [ ]:
# Load dataset dan tampilkan beberapa baris pertama
df = pd.read_csv('Churn_Modelling.csv')
display(df.head())

# Tampilkan informasi dasar: jumlah baris dan kolom, tipe data, statistik deskriptif, dan jumlah missing value
print("\n--- INFORMASI DASAR DATASET ---")
print(f"Jumlah baris dan kolom: {df.shape}")
print("\nTipe Data:")
print(df.dtypes)
print("\nStatistik Deskriptif:")
display(df.describe())
print(f"\nJumlah Missing Value:\n{df.isnull().sum()}")

In [ ]:
# Tampilkan distribusi kelas target (Exited) dalam bentuk bar plot. Perhatikan apakah dataset ini imbalanced.
plt.figure(figsize=(6, 4))
sns.countplot(x='Exited', data=df, palette='Set2')
plt.title('Distribusi Kelas Target (Exited)')
plt.xlabel('Exited (0 = Stay, 1 = Churn)')
plt.ylabel('Jumlah Nasabah')
plt.show()

# Cek persentase untuk melihat apakah dataset imbalanced
churn_counts = df['Exited'].value_counts(normalize=True) * 100
print(f"Persentase Nasabah Bertahan (0): {churn_counts[0]:.2f}%")
print(f"Persentase Nasabah Churn (1): {churn_counts[1]:.2f}% (Dataset tergolong imbalanced)")

In [ ]:
# Tampilkan distribusi fitur numerik (misalnya Age, CreditScore, Balance) menggunakan histogram atau KDE plot, dibedakan berdasarkan kelas Exited
numeric_cols = ['CreditScore', 'Age', 'Balance', 'EstimatedSalary']
for col in numeric_cols:
    plt.figure(figsize=(7, 3))
    sns.histplot(data=df, x=col, hue='Exited', kde=True, bins=30, palette='viridis', common_norm=False)
    plt.title(f'Distribusi {col} berdasarkan Exited')
    plt.xlabel(col)
    plt.ylabel('Frekuensi')
    plt.show()

In [ ]:
# Tampilkan proporsi churn untuk setiap kategori pada fitur kategorikal (Geography, Gender, IsActiveMember, NumOfProducts)
cat_cols = ['Geography', 'Gender', 'IsActiveMember', 'NumOfProducts']
for col in cat_cols:
    plt.figure(figsize=(7, 3))
    # Menghitung mean/proporsi churn per kategori
    sns.barplot(x=col, y='Exited', data=df, errorbar=None, palette='Blues_d')
    plt.title(f'Proporsi Churn Berdasarkan {col}')
    plt.xlabel(col)
    plt.ylabel('Rate Churn')
    plt.show()

## Lakukan Exploratory Data Analysis (EDA) tambahan yang diperlukan.

# 2. Preprocessing

ANN hanya menerima input numerik dan sangat sensitif terhadap skala fitur. Perhatikan jenis setiap kolom sebelum memilih preprocessing:

| Jenis kolom | Contoh | Preprocessing |
|---|---|---|
| Identitas (tidak informatif) | RowNumber, CustomerId, Surname | Drop |
| Kategorikal nominal | Geography | OneHotEncoder |
| Kategorikal biner | Gender | Encode menjadi 0/1 |
| Numerik | CreditScore, Age, Balance, dll. | StandardScaler |

In [ ]:
# Drop kolom yang tidak relevan untuk prediksi.
# Jelaskan secara singkat (dalam komentar) alasan kolom tersebut di-drop.
# Alasan: RowNumber, CustomerId, dan Surname di-drop karena bersifat identitas unik 
# dan tidak memiliki korelasi atau pengaruh langsung terhadap pola perilaku churn nasabah.
df_clean = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])

In [ ]:
# Pisahkan fitur (X) dan target (y).
X = df_clean.drop(columns=['Exited'])
y = df_clean['Exited'].values

# Bagi data menjadi train set, validation set, dan test set (misalnya 70:15:15).
# Gunakan stratify=y agar distribusi kelas tetap proporsional di setiap set.
# Validation set digunakan untuk memantau training (learning curve), test set hanya digunakan untuk evaluasi akhir.
from sklearn.model_selection import train_test_split

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=len(X_test)/len(X_train_val), random_state=42, stratify=y_train_val)

In [ ]:
# Lakukan encoding pada fitur kategorikal dan scaling pada fitur numerik.
# Fit encoder dan scaler hanya pada train set, kemudian transform pada train, validation, dan test set.
# Disarankan menggunakan ColumnTransformer agar rapi.
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
categorical_features = ['Geography', 'Gender']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first'), categorical_features)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

In [ ]:
# Pastikan X dan y untuk train, validation, dan test berupa numpy array bertipe float32.
# Contoh: X_train = np.asarray(X_train, dtype="float32")
import numpy as np

X_train = np.asarray(X_train_processed, dtype="float32")
X_val = np.asarray(X_val_processed, dtype="float32")
X_test = np.asarray(X_test_processed, dtype="float32")

y_train = np.asarray(y_train, dtype="float32")
y_val = np.asarray(y_val, dtype="float32")
y_test = np.asarray(y_test, dtype="float32")

# 3. Eksperimen Model

## 3.1 Baseline: Logistic Regression

Sebelum membangun ANN, buat model sederhana sebagai pembanding. Model yang lebih kompleks hanya layak digunakan jika performanya lebih baik dari baseline.

In [ ]:
# Inisialisasi dan latih model Logistic Regression (sklearn) menggunakan train set.
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train, y_train)

# Lakukan prediksi pada test set, simpan hasil prediksi label dan probabilitas kelas 1.
y_pred_lr = lr_model.predict(X_test)
y_prob_lr = lr_model.predict_proba(X_test)[:, 1]

## 3.2 ANN Sederhana

Bangun ANN dengan **1 hidden layer**. Karena ini klasifikasi biner, gunakan **1 neuron output dengan aktivasi Sigmoid** dan **Binary Cross-Entropy** sebagai loss function.

Contoh arsitektur: `Input → Dense(16, ReLU) → Dense(1, Sigmoid)`

In [ ]:
# Definisikan arsitektur ANN sederhana menggunakan keras.Sequential.
# Gunakan keras.Input(shape=(jumlah_fitur,)) sebagai layer pertama, lalu layers.Dense untuk hidden layer dan output layer.
# Tampilkan ringkasan model beserta jumlah parameternya dengan model.summary().
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

model_ann1 = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_ann1.summary()

In [ ]:
# Compile model: tentukan optimizer (misalnya keras.optimizers.Adam(learning_rate=...)), loss="binary_crossentropy", dan metrics=["accuracy"].
model_ann1.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                   loss='binary_crossentropy',
                   metrics=['accuracy'])

# Latih model dengan model.fit(), gunakan validation_data=(X_val, y_val) agar loss dan accuracy validation tercatat setiap epoch.
# Simpan hasil model.fit() ke variabel history.
history1 = model_ann1.fit(X_train, y_train,
                          validation_data=(X_val, y_val),
                          epochs=50,
                          batch_size=32,
                          verbose=1)

In [ ]:
# Tampilkan learning curve (loss dan accuracy) train vs validation dalam bentuk line plot.
# Gunakan history.history["loss"], history.history["val_loss"], history.history["accuracy"], dan history.history["val_accuracy"].
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history1.history['loss'], label='Train Loss')
plt.plot(history1.history['val_loss'], label='Val Loss')
plt.title('ANN Sederhana - Loss')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history1.history['accuracy'], label='Train Accuracy')
plt.plot(history1.history['val_accuracy'], label='Val Accuracy')
plt.title('ANN Sederhana - Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.show()

In [ ]:
# Lakukan prediksi pada test set dengan model.predict(X_test). Hasilnya berupa probabilitas kelas 1.
# Ubah menjadi label dengan threshold 0.5, lalu simpan probabilitas dan label untuk keperluan evaluasi.
y_prob_ann1 = model_ann1.predict(X_test).flatten()
y_pred_ann1 = (y_prob_ann1 >= 0.5).astype(int)

## 3.3 ANN Lebih Dalam dengan Dropout

Bangun ANN dengan **2–3 hidden layer** dan tambahkan **Dropout** di antara hidden layer untuk mengurangi overfitting.

In [ ]:
# Definisikan arsitektur ANN yang lebih dalam, tambahkan layers.Dropout(rate) di antara hidden layer.
# Tampilkan ringkasan model beserta jumlah parameternya.
from tensorflow.keras.layers import Dropout

model_ann2 = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model_ann2.summary()

In [ ]:
# Latih model dengan cara yang sama seperti 3.2, lalu tampilkan learning curve train vs validation.
# Bandingkan bentuk learning curve-nya dengan model 3.2. Apakah ada tanda overfitting?
model_ann2.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                   loss='binary_crossentropy',
                   metrics=['accuracy'])

history2 = model_ann2.fit(X_train, y_train,
                          validation_data=(X_val, y_val),
                          epochs=50,
                          batch_size=32,
                          verbose=1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history2.history['loss'], label='Train Loss')
plt.plot(history2.history['val_loss'], label='Val Loss')
plt.title('ANN Deep + Dropout - Loss')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history2.history['accuracy'], label='Train Accuracy')
plt.plot(history2.history['val_accuracy'], label='Val Accuracy')
plt.title('ANN Deep + Dropout - Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.show()

In [ ]:
# Lakukan prediksi pada test set.
# Simpan hasil prediksi label dan probabilitas kelas 1 untuk keperluan evaluasi.
y_prob_ann2 = model_ann2.predict(X_test).flatten()
y_pred_ann2 = (y_prob_ann2 >= 0.5).astype(int)

## 3.4 Penanganan Data Imbalanced

Pada data imbalanced, model cenderung lebih sering memprediksi kelas mayoritas sehingga recall kelas churn rendah. Salah satu cara mengatasinya adalah dengan memberi bobot lebih besar pada kelas minoritas di loss function.

In [ ]:
# Gunakan arsitektur terbaik dari 3.2 atau 3.3 (buat ulang modelnya agar bobot mulai dari awal), lalu latih dengan class weight.
# Hitung bobot setiap kelas berdasarkan train set, misalnya dengan sklearn.utils.class_weight.compute_class_weight("balanced", ...).
# Masukkan ke model.fit() melalui parameter class_weight={0: bobot_kelas_0, 1: bobot_kelas_1}.
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(class_weights))

model_ann3 = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model_ann3.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                   loss='binary_crossentropy',
                   metrics=['accuracy'])

history3 = model_ann3.fit(X_train, y_train,
                          validation_data=(X_val, y_val),
                          epochs=50,
                          batch_size=32,
                          class_weight=class_weights_dict,
                          verbose=1)

In [ ]:
# Tampilkan learning curve train vs validation.
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history3.history['loss'], label='Train Loss')
plt.plot(history3.history['val_loss'], label='Val Loss')
plt.title('ANN + Class Weight - Loss')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history3.history['accuracy'], label='Train Accuracy')
plt.plot(history3.history['val_accuracy'], label='Val Accuracy')
plt.title('ANN + Class Weight - Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.show()

# Lakukan prediksi pada test set, simpan hasil prediksi label dan probabilitas kelas 1.
y_prob_ann3 = model_ann3.predict(X_test).flatten()
y_pred_ann3 = (y_prob_ann3 >= 0.5).astype(int)

# 4. Evaluasi Model

In [ ]:
# Tampilkan metrik evaluasi untuk setiap model: Accuracy, Precision, Recall, dan F1-Score.
# Sajikan dalam satu tabel perbandingan agar mudah dibandingkan.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd

def evaluate_model(y_true, y_pred, y_prob):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1-Score': f1_score(y_true, y_pred, zero_division=0),
        'AUC': roc_auc_score(y_true, y_prob)
    }

results = {
    'Logistic Regression': evaluate_model(y_test, y_pred_lr, y_prob_lr),
    'ANN Sederhana': evaluate_model(y_test, y_pred_ann1, y_prob_ann1),
    'ANN + Dropout': evaluate_model(y_test, y_pred_ann2, y_prob_ann2),
    'ANN + Class Weight': evaluate_model(y_test, y_pred_ann3, y_prob_ann3)
}

df_results = pd.DataFrame(results).T
display(df_results)

In [ ]:
# Tampilkan Confusion Matrix setiap model dalam bentuk heatmap.
# Susun dalam satu baris subplot.
from sklearn.metrics import confusion_matrix
import seaborn as sns

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
preds = [y_pred_lr, y_pred_ann1, y_pred_ann2, y_pred_ann3]
titles = ['Logistic Regression', 'ANN Sederhana', 'ANN + Dropout', 'ANN + Class Weight']

for ax, pred, title in zip(axes, preds, titles):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Tampilkan ROC Curve seluruh model dalam satu plot beserta nilai AUC masing-masing.
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(8, 6))
models_roc = [
    ('Logistic Regression', y_prob_lr),
    ('ANN Sederhana', y_prob_ann1),
    ('ANN + Dropout', y_prob_ann2),
    ('ANN + Class Weight', y_prob_ann3)
]

for name, prob in models_roc:
    fpr, tpr, _ = roc_curve(y_test, prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

# 5. Analisis

### Jelaskan secara detail hasil analisis yang didapatkan dari eksperimen yang telah dilakukan. Bandingkan performa Logistic Regression dan setiap konfigurasi ANN berdasarkan hasil evaluasi. Bahas pula bentuk learning curve masing-masing model (overfitting/underfitting), pengaruh penambahan layer dan Dropout, serta pengaruh class weight terhadap precision dan recall kelas churn. Pastikan analisis didasari oleh hasil eksperimen sendiri yang telah dilakukan.

berdasarkan eksperimen yg udh dilakuin performa logistic regression kulihat cuman jd baseline krn dia linear jdi krn data nasabah bank bnyk pola kompleks dia kalah fleksibel sm ann yg lbh gampang nangkap pola non linear trus pas nyoba ann sederhana sm yg lbh deep trnyata model deep gampang bgt kena overfitting klo ga dipakein dropout makanya pas ditambahin dropout di hidden layer learning curvenya jd jauh lbh stabil antartrain sm val trus pas ngeliat data churn trnyata imbalanced bgt yg bikin model sering bias ke kelas mayoritas tp stlh dimasukin class weight model jd lbh peka ke kelas minoritas jd recall buat nasabah churn naiknya lumayan walau precisionnya agak turun dikit tp scr keseluruhan jauh lbh seimbang performanya

# 6. Kesimpulan dan Saran

### Berikan kesimpulan dari eksperimen yang telah dilakukan. Saran dapat berupa rekomendasi untuk eksperimen selanjutnya, perbaikan model, atau hal-hal lain yang relevan dengan tugas ini.

kesimpulannya model ann yg pake arsitektur deep ditambah dropout sm class weight lbh optimal bgt buat ngehandle kasus prediksi churn nasabah ini dbanding model baseline biasa saran bwt kdpannya mungkin bs dicoba tuning hyperparameter lg kaya nyesuaiin learning rate, nambah epoch, atau pake teknik resampling lain ky smote biar hasilnya makin mantap